# Module 9 — Serverless deep learning (AWS Lambda): commented walkthrough

Alexey's Module 9 code (lessons 9.1 – 9.7), with comments that explain **three things** for every step:

- `# GOAL`   what this step is for, in plain words
- `# WHY`    the machine-learning / deployment logic behind it (what could go wrong if we skipped it)
- `# PYTHON` how the Python / NumPy / Pillow / ONNX Runtime syntax works, so you can write it yourself later

The original, uncommented notebook is untouched in `../notebooks/`.

**Which material this follows, and why.** The lesson videos 9.3 – 9.6 use **TensorFlow Lite**. Alexey's own module README marks them *outdated* (no `tflite_runtime` wheels for current Python) and points to the **ONNX workshop** in `../notebooks/workshop/`. **HW9 uses ONNX Runtime**, not TF-Lite. So this notebook keeps the lessons' order and headings, but wherever the lecture runs TF-Lite it runs the workshop's ONNX code instead (`workshop/lambda-onnx/`, MobileNetV2 clothing model exported from PyTorch). That model takes the same kind of input as the HW9 model (channels-first, ImageNet mean/std), which is the part that matters for the homework. The TF-Lite lecture notebook stays in `../notebooks/09-serverless-lecture.ipynb` as reference only: it needs TensorFlow and the Keras `.h5` model, which are not installed/downloaded here.

| | Lecture 9.3 – 9.6 | Workshop (this notebook) | HW9 |
|---|---|---|---|
| inference runtime | `tflite_runtime` 2.14, custom wheel | `onnxruntime` | `onnxruntime`, every package pinned |
| model | Xception (Keras), 10 classes | MobileNetV2 (PyTorch → ONNX), 10 classes | hair classifier, 1 probability (`straight`) |
| input layout | (1, 299, 299, 3) channels-last | (1, 3, 224, 224) channels-first | channels-first, size: see Q2 |
| scaling | `x / 127.5 - 1` | `/255`, ImageNet mean/std | `/255`, ImageNet mean/std |
| resize | keras-image-helper (`NEAREST`) | keras-image-helper (`NEAREST`) | own Pillow code, **`BILINEAR`** |
| event key | `url` | `url` | `image_url` (direct or API Gateway `body`) |
| local port | 8080 | 8080 | 9000 |

Commented copies of the deployment files are in `commented/` (`lecture_tflite/`: `lambda_function.py`, `Dockerfile`, `test.py`; `workshop_onnx/`: `lambda_function.py`, `Dockerfile`). Section 9.4 imports the commented ONNX `lambda_function.py` and calls its handler like Lambda would.

**What was executed when this notebook was built:** every code cell, on CPU (onnxruntime 1.29, Pillow 12, NumPy 2, pandas 3, scikit-learn 1.9). **Not executed:** Docker build/run, ECR, Lambda and API Gateway. Those are terminal commands in markdown cells; the one code cell that calls a running container (9.5) prints a message instead of failing when no container is up.

Differences from the lecture:
- TF-Lite → ONNX Runtime (the workshop's path). `convert-model.py` and the interpreter calls are shown as text in 9.3, not run (they need TensorFlow);
- the images are read from the local `pants.jpg` (the image behind `http://bit.ly/mlbookcamp-pants`); 9.4 downloads it from its GitHub address because `bit.ly` is only a redirect;
- 9.2: the toy handlers are called as plain Python functions instead of from the AWS console, and the workshop's scikit-learn Lambda is rebuilt here: the model is trained from `data-week-3.csv` instead of unpickling the shipped `model.bin` (pickled with scikit-learn 1.7.1; loading it on 1.9 raises `InconsistentVersionWarning`). The workshop's `train.py` uses `df.dtypes == 'object'`, which selects **nothing** on pandas 3; here it is `select_dtypes(include=['object', 'string'])`. And its `DictVectorizer()` (sparse) + `LogisticRegression(solver='liblinear')` stops on scikit-learn 1.9 with *"Only sparse matrices with 32-bit integer indices are accepted"*; here it is `DictVectorizer(sparse=False)`;
- a hand-written Pillow preprocessing sits next to keras-image-helper, to show that the helper resizes with `NEAREST` and what `BILINEAR` (HW9's choice) changes;
- the two serialisation errors of lesson 9.5 (`float32 is not JSON serializable`) and the API Gateway event shapes of 9.7 are reproduced in code;
- the 9.6 pricing is computed in code from stated assumptions instead of read off the AWS page;
- `%autosave 0`, `!python -V` and `!pip install` are dropped (install `onnxruntime pillow keras-image-helper requests` once in your env).

## 9.1 Introduction to Serverless

The plan for the module: take the clothing classifier from Module 8 and serve it **without a server we manage**.
A client sends `{"url": "<image>"}`; AWS Lambda runs our function inside a Docker image (code + model + inference
runtime); the function downloads the image, preprocesses it, runs the model and returns the class scores as JSON.
API Gateway (9.7) puts an HTTP URL in front of it.

In [ ]:
# PYTHON: 'import X as Y' loads a library under a short nickname.
import json                 # the format Lambda speaks: events come in as JSON, results go out as JSON
import os                   # environment variables (MODEL_NAME) and file sizes
import pickle               # save / load the scikit-learn model (9.2)

import numpy as np          # the model input is a NumPy array
import pandas as pd         # only for the scikit-learn example in 9.2
from PIL import Image       # Pillow: open, convert and resize images (replaces keras.preprocessing)
import onnxruntime as ort   # the inference-only engine that replaces TF-Lite (9.3)

# The test image: the pants photo the lecture downloads from http://bit.ly/mlbookcamp-pants.
PANTS_FILE = 'pants.jpg'
PANTS_URL = ('https://raw.githubusercontent.com/alexeygrigorev/clothing-dataset-small/'
             'master/test/pants/4aabd82c-82e1-4181-a84d-d0c6e550d26d.jpg')

## 9.2 AWS Lambda

A Lambda function is **one Python function** with a fixed signature, `lambda_handler(event, context)`.
`event` is the JSON payload, already parsed into a `dict`. Whatever the function returns goes back to the caller,
converted to JSON. Because it is just a function, we can call it directly here, with no AWS involved.

In [ ]:
# GOAL: the lecture's first toy function, exactly as typed in the AWS console.
# WHY:  print() inside a Lambda goes to CloudWatch Logs: that is how you see what arrived in 'event'.
def lambda_handler(event, context):
    print("parameters: ", event)
    return "PONG"

# GOAL: "test" it like the console's Test button does: hand it an event dict. context is unused -> None.
lambda_handler({"url": "some-url-of-pants"}, None)

In [ ]:
# GOAL: the second toy version: read the parameter and return a dict, the shape a real prediction will have.
def lambda_handler(event, context):
    print("parameters: ", event)
    url = event['url']
    # In the real function this line becomes: result = predict(url)
    return {"prediction": "pants"}

result = lambda_handler({"url": "some-url-of-pants"}, None)

# WHY:  Lambda must turn the return value into JSON. json.dumps is that step: if it fails here, it fails on AWS
#       ("Unable to marshal response").
json.dumps(result)

### Serverless vs serverful

With Lambda we pay **per request and per millisecond of run time**, nothing while idle; AWS starts more copies
when traffic grows. The price: a **cold start** (download the image, import libraries, load the model) on the first
request after a quiet period. With Elastic Beanstalk / EC2 (Module 5) a machine runs, and bills, all the time.

### Workshop: a scikit-learn model behind Lambda

The workshop starts with the Module 3 churn model, because it shows the whole contract with no deep learning in
the way. (A plain ZIP upload is limited to 250 MB unpacked, and scikit-learn + SciPy + NumPy exceed it:
that is why everything later goes into a **Docker image**, limit 10 GB.)

In [ ]:
# GOAL: rebuild the workshop's model.bin: DictVectorizer + LogisticRegression in one Pipeline, trained on all data.
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv('data-week-3.csv')
df.columns = df.columns.str.lower().str.replace(' ', '_')

# WHY: the workshop's train.py uses df.dtypes == 'object', which selects NO columns on pandas 3 (text = 'str'),
#      so nothing is lower-cased and every churn label becomes 0. select_dtypes works on both pandas 2 and 3.
categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce').fillna(0)
df.churn = (df.churn == 'yes').astype(int)

numerical = ['tenure', 'monthlycharges', 'totalcharges']
categorical = [
    'gender', 'seniorcitizen', 'partner', 'dependents',
    'phoneservice', 'multiplelines', 'internetservice',
    'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
    'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod',
]

# WHY: a Pipeline is ONE object holding both steps, so the Lambda loads one file and cannot pair the model
#      with the wrong vectorizer.
# PYTHON: make_pipeline(a, b) chains the steps; .fit(dicts, y) runs fit_transform on a, then fit on b.
# NOTE: the workshop writes DictVectorizer() (sparse output). On scikit-learn 1.9 'liblinear' rejects that sparse
#       matrix ("Only sparse matrices with 32-bit integer indices are accepted"), so we ask for a dense array.
pipeline = make_pipeline(DictVectorizer(sparse=False), LogisticRegression(solver='liblinear'))
pipeline.fit(df[categorical + numerical].to_dict(orient='records'), df.churn)

# PYTHON: 'wb' = write bytes; pickle.dump serialises the whole Python object into the file.
with open('model.bin', 'wb') as f_out:
    pickle.dump(pipeline, f_out)

df.churn.mean()    # sanity check: ~0.27. 0.0 would mean the text cleaning above did not run.

In [ ]:
# GOAL: the workshop's lambda-sklearn/lambda_function.py, line for line.
# WHY: the model is loaded at MODULE level, once per container. Lambda reuses a warm container for the next
#      requests, so only the first request (the cold start) pays for loading.
with open('model.bin', 'rb') as f_in:
    pipeline = pickle.load(f_in)

def predict_single(customer):
    # PYTHON: predict_proba expects a LIST of records; the pipeline's DictVectorizer accepts a single dict too.
    #         [0, 1] = first row, column of class 1 (churn).
    result = pipeline.predict_proba(customer)[0, 1]
    return float(result)

def lambda_handler(event, context):
    customer = event['customer']
    prob = predict_single(customer)

    # WHY: float() and bool() turn NumPy scalars into plain Python types, which json can serialise (next cell).
    return {
        "churn_probability": prob,
        "churn": bool(prob >= 0.5)
    }

In [ ]:
# GOAL: the workshop's customer.json event, sent to the handler as the console's Test button would.
event = {
    "customer": {
        "gender": "female", "seniorcitizen": 0, "partner": "yes", "dependents": "no",
        "phoneservice": "no", "multiplelines": "no_phone_service", "internetservice": "dsl",
        "onlinesecurity": "no", "onlinebackup": "yes", "deviceprotection": "no", "techsupport": "no",
        "streamingtv": "no", "streamingmovies": "no", "contract": "month-to-month",
        "paperlessbilling": "yes", "paymentmethod": "electronic_check",
        "tenure": 1, "monthlycharges": 29.85, "totalcharges": 29.85,
    }
}

result = lambda_handler(event, None)
json.dumps(result)    # serialises cleanly

In [ ]:
# GOAL: see why bool() is there. (prob >= 0.5) on a NumPy float gives np.bool_, and json refuses it.
# PYTHON: try / except catches the error so the notebook keeps running; 'as e' keeps the error to print it.
raw_prob = pipeline.predict_proba(event['customer'])[0, 1]     # np.float64, not float
try:
    json.dumps({"churn": raw_prob >= 0.5})
except TypeError as e:
    print('TypeError:', e)

## 9.3 TensorFlow Lite (→ ONNX Runtime)

**Why not TensorFlow itself:** ~1.7 GB installed, slow to import, big RAM footprint. On Lambda all of that is
paid on every cold start and in image storage. We only need **inference** (`model.predict`), not training.

**What the lecture does (TF-Lite, not run here):**

```python
converter = tf.lite.TFLiteConverter.from_keras_model(model)       # convert-model.py
tflite_model = converter.convert()
with open('clothing-model.tflite', 'wb') as f_out:
    f_out.write(tflite_model)

interpreter = tflite.Interpreter(model_path='clothing-model.tflite')
interpreter.allocate_tensors()
input_index = interpreter.get_input_details()[0]['index']
output_index = interpreter.get_output_details()[0]['index']
interpreter.set_tensor(input_index, X); interpreter.invoke(); preds = interpreter.get_tensor(output_index)
```

**What we do instead (workshop):** **ONNX** (Open Neural Network Exchange) is a framework-neutral file format for a
trained network; **ONNX Runtime** runs it. Keras models get there via `tf2onnx` (`workshop/lambda-keras/convert/`),
PyTorch models with `torch.onnx.export(...)`. The conversion is done once, offline; the Lambda only needs
`onnxruntime` (tens of MB, ordinary wheels for every platform). Same idea as TF-Lite, newer and more portable tool.

In [ ]:
# GOAL: load the ONNX model: the equivalent of tflite.Interpreter(...) + allocate_tensors().
# WHY:  providers=['CPUExecutionProvider']: Lambda has no GPU. Naming it explicitly keeps the result the same on
#       your Mac, in Docker and on AWS.
MODEL_FILE = 'clothing_classifier_mobilenet_v2_latest.onnx'
session = ort.InferenceSession(MODEL_FILE, providers=['CPUExecutionProvider'])

# PYTHON: os.path.getsize returns bytes; // (1024 * 1024) = whole megabytes. The lecture's Xception .h5 is ~85 MB.
os.path.getsize(MODEL_FILE) // (1024 * 1024), 'MB'

In [ ]:
# GOAL: the ONNX equivalent of get_input_details() / get_output_details(): what does the graph expect and return?
# WHY:  ONNX Runtime feeds inputs and fetches outputs BY NAME, and the shape tells you the layout:
#       (batch, 3, H, W) = channels-first (PyTorch); (batch, H, W, 3) = channels-last (Keras).
#       A symbolic first dimension (a string like 'batch_size') means "any batch size".
# PYTHON: get_inputs() returns a list of NodeArg objects with .name, .shape and .type attributes.
for node in session.get_inputs():
    print('input: ', node.name, node.shape, node.type)
for node in session.get_outputs():
    print('output:', node.name, node.shape, node.type)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

### Preprocessing without TensorFlow

The lecture replaces `keras.preprocessing.image.load_img` + `preprocess_input` with Pillow + NumPy. Same here, for
the PyTorch-style model: resize to 224 × 224, scale to [0, 1], subtract the ImageNet mean, divide by the std, move
the channels first, add a batch dimension, `float32`. Every one of these choices is part of the model's interface:
change one and the model sees numbers it was never trained on.

In [ ]:
# GOAL: preprocess the pants photo by hand, step by step.
# PYTHON: 'with' closes the file afterwards. .convert('RGB') guarantees 3 channels (PNGs can have 4, greyscale 1).
with Image.open(PANTS_FILE) as img:
    img = img.convert('RGB')
    # WHY: NEAREST = what keras-image-helper (the workshop's helper) uses. We compare with BILINEAR below.
    # PYTHON: Pillow's size is (width, height); NumPy's shape will be (height, width, channels).
    img_nearest = img.resize((224, 224), Image.Resampling.NEAREST)
    img_bilinear = img.resize((224, 224), Image.Resampling.BILINEAR)

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)   # ImageNet per-channel mean (R, G, B)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)    # ImageNet per-channel std

def to_tensor(small_img):
    # PYTHON: np.asarray(img, dtype=np.float32) -> (224, 224, 3) array of pixel values 0..255.
    x = np.asarray(small_img, dtype=np.float32) / 255.0
    # PYTHON: broadcasting: the (3,) MEAN/STD apply to the LAST axis (channels) of every pixel.
    x = (x - MEAN) / STD
    # WHY:    the graph is channels-first: (H, W, C) -> (C, H, W) ...
    # PYTHON: transpose((2, 0, 1)) puts old axis 2 (channels) first; [None, ...] adds the batch axis in front.
    return np.transpose(x, (2, 0, 1))[None, ...]

X = to_tensor(img_nearest)
X.shape, X.dtype

In [ ]:
# GOAL: run the model: the equivalent of set_tensor / invoke / get_tensor.
# PYTHON: session.run([output names], {input name: array}) returns a LIST with one array per requested output.
result = session.run([output_name], {input_name: X})
preds = result[0][0]        # first output, first (only) image in the batch -> 10 scores

classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

# PYTHON: dict(zip(a, b)) pairs names with scores; .tolist() gives plain floats (easier to read, JSON-safe).
dict(zip(classes, preds.tolist()))

In [ ]:
# GOAL: these are LOGITS (raw scores, any sign). Softmax turns them into probabilities that sum to 1.
# WHY:  the clothing model has 10 classes and returns logits; the HW9 model ends in a sigmoid and already
#       returns ONE probability. Always check what the output means before you threshold it.
# PYTHON: subtracting the max before exp() is the standard trick against overflow; the result is unchanged.
exp = np.exp(preds - preds.max())
probs = exp / exp.sum()
classes[int(np.argmax(probs))], probs.max().round(3)

In [ ]:
# GOAL: the workshop's way: keras-image-helper with a custom preprocessing function (workshop/lambda-onnx).
# WHY:  check that our hand-written version matches it exactly, which proves the helper resizes with NEAREST.
from keras_image_helper import create_preprocessor

def preprocess_pytorch_style(X):
    X = X / 255.0
    mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)
    X = X.transpose(0, 3, 1, 2)          # NHWC -> NCHW
    X = (X - mean) / std
    return X.astype(np.float32)

preprocessor = create_preprocessor(preprocess_pytorch_style, target_size=(224, 224))
X_helper = preprocessor.from_path(PANTS_FILE)

# PYTHON: np.allclose compares element-wise within a tiny tolerance (float rounding).
np.allclose(X_helper, X, atol=1e-5)

In [ ]:
# GOAL: same photo, BILINEAR instead of NEAREST. How much do the input and the output move?
# WHY:  NEAREST copies one source pixel per target pixel; BILINEAR averages neighbours. On a downscale from
#       400 x 534 to 224 x 224 that changes many pixel values, so the model output changes too. Here pants still
#       wins; on a single-probability model like HW9's the shift can cross the 0.5 threshold. HW9 therefore pins
#       BILINEAR explicitly: don't reuse keras-image-helper there.
X_bil = to_tensor(img_bilinear)
preds_bil = session.run([output_name], {input_name: X_bil})[0][0]

print('max |pixel difference|  :', np.abs(X_bil - X).max().round(3))
print('pants logit NEAREST     :', preds[4].round(3))
print('pants logit BILINEAR    :', preds_bil[4].round(3))

In [ ]:
# GOAL: the two errors ONNX Runtime gives when the input breaks the interface.
# WHY:  the error text tells you which rule you broke: wrong dtype vs wrong layout.
try:
    session.run([output_name], {input_name: X.astype(np.float64)})       # float64 instead of float32
except Exception as e:
    print('dtype :', str(e).splitlines()[0])

try:
    session.run([output_name], {input_name: np.transpose(X, (0, 2, 3, 1))})  # channels-last instead of first
except Exception as e:
    print('layout:', str(e).splitlines()[0])

## 9.4 Preparing the code for Lambda

The lecture turns the notebook into a script (`jupyter nbconvert --to script`), deletes everything that is not
needed at prediction time, wraps the prediction in `predict(url)` and adds `lambda_handler`. The cleaned ONNX
version is `commented/workshop_onnx/lambda_function.py`. Here we **import that file** and call its handler,
which is exactly what the Lambda runtime does inside the container.

In [ ]:
# GOAL: import commented/workshop_onnx/lambda_function.py as a module.
# WHY:  the file reads the model name from the MODEL_NAME environment variable (the Dockerfile sets it) and opens
#       it relative to the current folder, so we set the variable first. Importing runs the module-level code:
#       the ONNX session is created now, once, like a Lambda cold start.
# PYTHON: sys.path is the list of folders Python searches for imports; insert(0, ...) puts ours first.
import sys
sys.dont_write_bytecode = True      # PYTHON: no __pycache__/ folder left next to the imported file

os.environ['MODEL_NAME'] = MODEL_FILE
sys.path.insert(0, 'commented/workshop_onnx')
import lambda_function

lambda_function.model_name, lambda_function.input_name, lambda_function.output_name

In [ ]:
# GOAL: invoke the handler with the same event the console / test.py would send.
# WHY:  the function downloads the image itself: the client sends a URL, not the pixels.
#       (Needs internet. The lecture's bit.ly link works too; this is its target.)
event = {'url': PANTS_URL}
response = lambda_function.lambda_handler(event, None)

# PYTHON: max(d, key=d.get) returns the KEY with the largest value.
max(response, key=response.get), json.dumps(response)[:80] + '...'

In [ ]:
# GOAL: reproduce lesson 9.5's "Object of type float32 is not JSON serializable" and its fix.
# WHY:  ONNX Runtime and TF-Lite return np.float32 arrays. Returning them from the handler makes Lambda fail
#       AFTER the prediction worked. .tolist() converts to plain Python floats.
scores = lambda_function.session.run([lambda_function.output_name], {lambda_function.input_name: X})[0][0]

try:
    json.dumps(dict(zip(classes, scores)))            # values are np.float32
except TypeError as e:
    print('TypeError:', e)

json.dumps(dict(zip(classes, scores.tolist())))[:60] + '...'   # fixed

## 9.5 Preparing a Docker image

AWS publishes **Lambda base images** (`public.ecr.aws/lambda/python:<version>`). They contain the runtime that calls
`lambda_handler` and an **emulator**, so `docker run` on your Mac answers HTTP requests exactly like the deployed
function. The commented Dockerfiles: `commented/lecture_tflite/Dockerfile` (the outdated TF-Lite one: Python 3.10,
a custom wheel, the glibc error of 9.5) and `commented/workshop_onnx/Dockerfile` (ONNX, plain `pip install`).

In [ ]:
# GOAL: read the ONNX Dockerfile with its comments. (! runs a shell command from the notebook.)
!cat commented/workshop_onnx/Dockerfile

Build and run it in a terminal (Docker Desktop must be running). The build context is the original workshop
folder, where the model file already sits next to `lambda_function.py`:

```bash
cd ../notebooks/workshop/lambda-onnx
docker build -t clothing-lambda-onnx .
docker run -it --rm -p 8080:8080 clothing-lambda-onnx
```

`-p 8080:8080` = host port : container port. The emulator always listens on 8080 **inside**; the number before the
colon is what you call from the Mac. Then run the next cell (it is `test.py`).

In [ ]:
# GOAL: test.py: POST the event to the local emulator.
# WHY:  the path /2015-03-31/functions/function/invocations is fixed: it imitates the AWS Invoke API, and the
#       request body becomes the handler's 'event'.
import requests

url = 'http://localhost:8080/2015-03-31/functions/function/invocations'
data = {'url': PANTS_URL}

try:
    result = requests.post(url, json=data, timeout=60).json()
    print(result)
except requests.exceptions.ConnectionError:
    print('No container is listening on localhost:8080: start it with the docker run command above.')

## 9.6 Creating the lambda function

Terminal steps (AWS CLI configured in the November setup session; `eu-west-1` as in the course, any region works):

```bash
# 1. a private registry for the image (once)
aws ecr create-repository --repository-name clothing-lambda-onnx --region eu-west-1
ECR_URL=<account-id>.dkr.ecr.eu-west-1.amazonaws.com          # from "repositoryUri" in the reply

# 2. log Docker in to ECR, tag the local image with the remote name, push
aws ecr get-login-password --region eu-west-1 | docker login --username AWS --password-stdin ${ECR_URL}
REMOTE_IMAGE=${ECR_URL}/clothing-lambda-onnx:v1
docker tag clothing-lambda-onnx ${REMOTE_IMAGE}
docker push ${REMOTE_IMAGE}
```

3. Console → Lambda → **Create function** → *Container image* → pick the image.
   **Architecture must match the image**: built on an Apple Silicon Mac it is `arm64` (choose arm64, which is also
   cheaper), or build with `docker build --platform linux/amd64 ...` and keep `x86_64`.
4. Configuration → General: **timeout 30 s** (default 3 s is too short for a cold start that loads a model and
   downloads an image), **memory 1024 MB**. Test with `{"url": "..."}`.
5. Updates later: push a new tag, then `aws lambda update-function-code --function-name <name> --image-uri ${REMOTE_IMAGE}`.

**Clean up when done** (the Lambda itself costs nothing while idle; the ECR image is the part that keeps billing):

```bash
aws lambda delete-function --function-name <name>
aws ecr delete-repository --repository-name clothing-lambda-onnx --force
aws apigateway delete-rest-api --rest-api-id <api-id>        # after 9.7
```

In [ ]:
# GOAL: the lesson's pricing question as arithmetic. Lambda bills GB-seconds (memory x duration) + requests.
# WHY:  it shows why serverless suits a spiky, low-traffic model: idle time is free.
# ASSUMPTIONS (check the AWS Lambda pricing page for your region; these are the us-east-1 / eu-west-1 list prices):
price_per_gb_second = {'x86_64': 0.0000166667, 'arm64': 0.0000133334}   # USD
price_per_million_requests = 0.20                                       # USD
free_gb_seconds, free_requests = 400_000, 1_000_000                     # monthly free tier

memory_gb = 1.0          # 1024 MB, as configured in 9.6
duration_s = 0.5         # a warm request; a cold start can take several seconds
requests_per_month = 10_000

gb_seconds = memory_gb * duration_s * requests_per_month
for arch, price in price_per_gb_second.items():
    # PYTHON: max(0, ...) = nothing to pay while usage stays under the free tier.
    cost = (max(0, gb_seconds - free_gb_seconds) * price
            + max(0, requests_per_month - free_requests) / 1e6 * price_per_million_requests)
    print(f'{arch:7s} {gb_seconds:,.0f} GB-s/month -> ${cost:.2f} after the free tier '
          f'(${gb_seconds * price:.2f} without it)')

## 9.7 API Gateway: exposing the lambda function

Console → API Gateway → **REST API** → resource `/predict` → method `POST` → integration *Lambda Function*
(the lecture leaves "proxy integration" **off**) → **Deploy API** to a stage (`test`) → URL
`https://<id>.execute-api.<region>.amazonaws.com/test/predict`. Swap it into `test.py`.
The endpoint is open to anyone with the URL: fine for learning, delete it afterwards.

**Two event shapes.** Without proxy integration (the lecture), the handler's `event` *is* the request body.
With **proxy** integration (HTTP APIs, Function URLs) the body arrives as a **JSON string** under `event['body']`,
and the handler must return `{"statusCode": ..., "body": "<json string>"}`. HW9's handler accepts both.

In [ ]:
# GOAL: the two events API Gateway can send for the same request body.
body = {'url': PANTS_URL}

event_non_proxy = body                                   # the lecture's setup
event_proxy = {                                          # proxy integration (trimmed: real ones carry more keys)
    'httpMethod': 'POST',
    'path': '/predict',
    'headers': {'Content-Type': 'application/json'},
    'body': json.dumps(body),                            # a STRING, not a dict
}

# WHY: the workshop handler reads event['url'] and fails on the proxy shape.
try:
    lambda_function.lambda_handler(event_proxy, None)
except KeyError as e:
    print('KeyError:', e)

In [ ]:
# GOAL: a handler that accepts both shapes and answers in the proxy format.
# PYTHON: isinstance(x, str) checks the type; json.loads parses a JSON string into a dict.
def lambda_handler_any(event, context):
    payload = event
    if isinstance(event.get('body'), str):
        payload = json.loads(event['body'])

    result = lambda_function.predict(payload['url'])
    return {
        'statusCode': 200,
        'headers': {'Content-Type': 'application/json'},
        'body': json.dumps(result),                      # the body goes back as a string, too
    }

response = lambda_handler_any(event_proxy, None)

# WHY: the CLIENT has to json.loads() the body again to get the numbers back.
scores_back = json.loads(response['body'])
response['statusCode'], max(scores_back, key=scores_back.get)

## 9.8 Summary

| Piece | What it does | Watch out |
|---|---|---|
| `lambda_handler(event, context)` | the entry point; `event` = parsed JSON | return only JSON-serialisable types (`.tolist()`, `float()`, `bool()`) |
| module-level model loading | runs once per container (cold start) | never load the model inside the handler |
| TF-Lite / **ONNX Runtime** | inference without full TensorFlow | TF-Lite is outdated for current Python; ONNX is the current path |
| preprocessing | resize, scale, normalise, layout, dtype | it is part of the model: resize method, NCHW vs NHWC, `float32` |
| Lambda base image | runtime + local emulator, same image locally and on AWS | host port in `-p HOST:8080`; image architecture = function architecture |
| ECR → Lambda | registry + function from a container image | timeout 30 s, memory 1 GB; delete the repository afterwards |
| API Gateway | public HTTP URL in front of the function | proxy events carry the body as a string; the endpoint is open |

## 9.9 Explore more

- The same handler on Google Cloud Run / Azure Functions
- Serve a scikit-learn or XGBoost model as a Lambda with Docker (the workshop's `lambda-sklearn/`, with `uv`)
- Lambda **Function URLs**: an HTTPS endpoint without API Gateway (proxy-shaped events)
- Convert your Module 8 Keras model with `tf2onnx` (`workshop/lambda-keras/convert/`) and compare its outputs to Keras

## 9.10 Python 3.12 vs TF Lite 2.17

The lesson pins Python 3.10, NumPy 1.23 and a TF-Lite 2.14 wheel to keep the lecture stack alive. That fragility is
the reason the workshop, and HW9, moved to ONNX Runtime.